# Diffraction Analysis

Optics TP5 — L3 Physics, Sorbonne Université

Two experiments:
- Single slit diffraction -> measure slit width from minima positions
- Diffraction grating -> measure grating constant from mercury spectral lines

Single slit minima: $x_p = \frac{p \lambda f'}{a}$  
Grating (1st order): $\sin(\theta) = \frac{\lambda}{d}$

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from optics import diffraction

---
## Part 1: Single slit

Sodium lamp (lam = 578 nm), observation lens f' = 30 cm. We measure the positions of diffraction minima on a screen.

In [ ]:
p_data = np.array([-3, -2, -1, 1, 2, 3])

# measured positions in mm, converted to m
x_diff_data = np.array([-1.351, -0.933, -0.415, 0.451, 0.887, 1.358]) * 1e-3

lam = 578e-9  # m
D = 0.30      # m

x_diff_erreurs = np.full_like(x_diff_data, 5e-7)  # +/-0.5 um

print(f"{len(p_data)} minima measured")
print(f"Position range: {x_diff_data.min()*1e3:.3f} to {x_diff_data.max()*1e3:.3f} mm")

In [ ]:
plt.figure(figsize=(8, 5))
plt.errorbar(p_data, x_diff_data*1e3, yerr=x_diff_erreurs*1e3,
             fmt='o', capsize=4, label='measured minima')
plt.xlabel('Order p')
plt.ylabel('Position (mm)')
plt.title('Diffraction minima positions')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# linear as expected

### Fit to get slit width

From $x_p = \frac{\lambda f'}{a} \cdot p$, linear fit gives us $a$.

In [ ]:
res = diffraction.analyze_single_slit(x_diff_data, p_data, D, lam)

print(f"Slit width: a = {res['a']*1e6:.1f} +/- {res['a_err']*1e6:.1f} um")
print(f"R2 = {res['R2']:.6f}")

In [ ]:
# plot data + fit + residuals
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7),
                               gridspec_kw={'height_ratios': [3, 1]},
                               sharex=True)

ax1.errorbar(p_data, x_diff_data*1e3, yerr=x_diff_erreurs*1e3,
             fmt='o', capsize=4, label='data')

p_fit = np.linspace(p_data.min(), p_data.max(), 100)
x_fit = res['slope'] * p_fit + res['intercept']
ax1.plot(p_fit, x_fit*1e3, 'r-', lw=2, label='linear fit')

ax1.set_ylabel('Position (mm)')
ax1.set_title(f"Single slit diffraction -- a = {res['a']*1e6:.1f} um")
ax1.legend()
ax1.grid(alpha=0.3)

# residuals
x_predicted = res['slope'] * p_data + res['intercept']
residuals = (x_diff_data - x_predicted) * 1e3

ax2.errorbar(p_data, residuals, yerr=x_diff_erreurs*1e3, fmt='o', capsize=3)
ax2.axhline(0, color='red', ls='--', lw=1)
ax2.set_xlabel('Order p')
ax2.set_ylabel('Residual (mm)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual std: {np.std(residuals):.4f} mm")

R2 > 0.999 — the linear model works very well here.

---
## Part 2: Diffraction grating

Mercury lamp, screen at 20 cm. We measure first-order positions for 4 spectral lines.

In [ ]:
# mercury spectral lines
lambda_data = np.array([546.07, 404.66, 435.83, 365.02])  # nm
colors = ['green', 'violet', 'indigo', 'UV']

# first order positions (cm -> m)
x_data = np.array([3.4, 2.3, 2.6, 1.9]) * 1e-2

L = 20e-2  # screen distance (m)

# sin(theta) from geometry
sin_theta = x_data / np.sqrt(L**2 + x_data**2)

for c, wl, x in zip(colors, lambda_data, x_data):
    print(f"{c:8s}: {wl:.2f} nm, x = {x*100:.1f} cm")

In [ ]:
# fit sin(theta) = (1/d) * lambda
lambda_m = lambda_data * 1e-9

d, d_err = diffraction.get_grating_constant(sin_theta, lambda_m)

print(f"d = {d*1e6:.3f} +/- {d_err*1e6:.3f} um")
print(f"Grating density: {1/(d*1e-3):.0f} lines/mm")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

point_colors = ['green', 'violet', 'indigo', 'purple']
for wl, st, c, name in zip(lambda_data, sin_theta, point_colors, colors):
    ax.plot(wl, st, 'o', ms=10, color=c,
            markeredgecolor='black', markeredgewidth=1,
            label=f'{name} ({wl:.0f} nm)')

lam_fit = np.linspace(lambda_data.min()-20, lambda_data.max()+20, 100)
ax.plot(lam_fit, lam_fit * 1e-9 / d, 'r-', lw=2,
        label=f'fit: d = {d*1e6:.2f} um', alpha=0.7)

ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('sin(theta)')
ax.set_title('Grating: 1st order diffraction')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

sin(theta) is proportional to lambda as expected. Only 4 data points so the fit is not super constrained, but the result looks reasonable.

## Conclusion

**Single slit**: slit width determined with R2 > 0.999, the x vs p relationship is very clearly linear.

**Grating**: grating constant measured from 4 mercury lines. The sin(theta) vs lambda relationship holds, though more spectral lines would make the fit more robust.

---
*Optical Analysis Toolkit — data from L3 Physics TP5, Sorbonne Université*